# Arabic Sign Language Word Training (KArSL-502 to BiLSTM)

**Purpose:** Download KArSL-502 dataset from Kaggle, extract MediaPipe hand landmarks,
build 30-frame sequences, and train a BiLSTM model for ArSL word recognition.

**Output:** `arsl_word_sequences.npz` + `arsl_word_lstm_model_best.h5`

---

## 1. Setup & Configuration

In [1]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES & CONFIGURE
# ============================================================
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['kagglehub', 'mediapipe', 'scikit-learn', 'tqdm']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        print(f'Installing {pkg}...')
        install(pkg)

import os, glob, cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
import kagglehub
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# --- GPU Configuration ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print(f'GPU mode: {len(gpus)} GPU(s), mixed_float16')
else:
    print('CPU mode')

# --- Paths ---
PROJECT_DIR = Path('.').resolve()
OUTPUT_NPZ = PROJECT_DIR / 'arsl_word_sequences.npz'
MODEL_OUTPUT = PROJECT_DIR / 'arsl_word_lstm_model_best.h5'

# --- Hyperparameters ---
SEQUENCE_LENGTH = 30     # frames per sequence
NUM_FEATURES = 63        # 21 landmarks * 3 coords
MAX_VIDEOS_PER_CLASS = 50
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
print(f'Project: {PROJECT_DIR}')
print('Configuration loaded')


Installing scikit-learn...
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 Laptop GPU, compute capability 8.6
GPU mode: 1 GPU(s), mixed_float16
Project: C:\Users\HADEEL GAMALELDIN\Downloads\Letters_ORIGINAL
Configuration loaded


## 2. Download KArSL-502 from Kaggle

Uses `kagglehub` to download the dataset. You need Kaggle credentials
configured (either `~/.kaggle/kaggle.json` or environment variables).

In [2]:
# ============================================================
# CELL 2: DOWNLOAD KArSL-502 DATASET
# ============================================================

print('Downloading KArSL-502 from Kaggle...')
print('(This may take a while for first download)')

karsl_path = kagglehub.dataset_download('yousefdotpy/karsl-502')
karsl_path = Path(karsl_path)

print(f'Dataset path: {karsl_path}')

# Discover dataset structure
print(f'\nTop-level contents:')
for item in sorted(karsl_path.iterdir())[:20]:
    kind = 'DIR' if item.is_dir() else 'FILE'
    print(f'  [{kind}] {item.name}')

# Find all video files recursively
video_extensions = {'.mp4', '.avi', '.mov', '.mkv'}
all_videos = []
for ext in video_extensions:
    all_videos.extend(karsl_path.rglob(f'*{ext}'))

print(f'\nTotal video files found: {len(all_videos)}')
if all_videos:
    print(f'Sample path: {all_videos[0]}')


(This may take a while for first download)


  0%|          | 64.0M/23.7G [00:23<2:26:03, 2.90MB/s]


KeyboardInterrupt: 

## 3. Organize Videos by Sign Class

In [ ]:
# ============================================================
# CELL 3: PARSE DATASET STRUCTURE & BUILD CLASS MAPPING
# ============================================================

def discover_classes(dataset_path, videos):
    """
    Auto-discover class structure from KArSL folder hierarchy.
    KArSL typically organizes as: Category/SignID/videos
    or directly as SignName/videos.
    """
    class_videos = {}  # class_name -> [video_paths]
    
    for v in videos:
        # Use parent folder name as class label
        class_name = v.parent.name
        if class_name not in class_videos:
            class_videos[class_name] = []
        class_videos[class_name].append(v)
    
    return class_videos

class_videos = discover_classes(karsl_path, all_videos)
print(f'Discovered {len(class_videos)} sign classes')

# Sort by number of samples
sorted_classes = sorted(class_videos.items(), key=lambda x: len(x[1]), reverse=True)

print(f'\nTop 15 classes by sample count:')
for name, vids in sorted_classes[:15]:
    print(f'  {name}: {len(vids)} videos')

# Build class_id mapping
# Use all classes (or filter if needed)
class_names = [name for name, _ in sorted_classes]
class_to_id = {name: i for i, name in enumerate(class_names)}
id_to_class = {i: name for name, i in class_to_id.items()}
num_classes = len(class_to_id)

print(f'\nTotal classes: {num_classes}')
total_vids = sum(min(len(v), MAX_VIDEOS_PER_CLASS) for v in class_videos.values())
print(f'Total videos to process (capped): {total_vids}')


## 4. MediaPipe Landmark Extraction

In [ ]:
# ============================================================
# CELL 4: EXTRACT MEDIAPIPE HAND LANDMARKS FROM VIDEOS
# ============================================================

mp_hands = mp.solutions.hands

def extract_landmarks_from_video(video_path, hands_detector):
    """
    Extract per-frame hand landmarks from a video.
    Returns list of 63-dim arrays. Uses forward-fill for missing frames.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    
    all_landmarks = []
    last_valid = None
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands_detector.process(rgb)
        
        if results.multi_hand_landmarks:
            hand = results.multi_hand_landmarks[0]
            lm = np.array([[p.x, p.y, p.z] for p in hand.landmark]).flatten()
            # Wrist-relative normalization
            wrist = lm[:3].copy()
            for i in range(21):
                lm[i*3:(i+1)*3] -= wrist
            last_valid = lm
            all_landmarks.append(lm)
        elif last_valid is not None:
            all_landmarks.append(last_valid.copy())
    
    cap.release()
    return all_landmarks if len(all_landmarks) >= 5 else None

print('Extraction function defined')


## 5. Build 30-Frame Sequences & Save NPZ

In [ ]:
# ============================================================
# CELL 5: RESAMPLE TO FIXED LENGTH & BUILD DATASET
# ============================================================

def resample_sequence(landmarks_list, target_length=30):
    """Resample variable-length sequence to fixed length."""
    n = len(landmarks_list)
    if n == target_length:
        return np.array(landmarks_list)
    indices = np.linspace(0, n - 1, target_length).astype(int)
    return np.array([landmarks_list[i] for i in indices])

# Process all videos
all_X = []
all_y = []
all_signer_ids = []  # preserve signer info for signer-aware splitting
skipped = 0

with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    model_complexity=0,
    min_detection_confidence=0.5
) as hands:
    for class_name, videos in tqdm(class_videos.items(), desc='Processing classes'):
        class_id = class_to_id[class_name]
        
        for vpath in videos[:MAX_VIDEOS_PER_CLASS]:
            landmarks = extract_landmarks_from_video(vpath, hands)
            if landmarks is None:
                skipped += 1
                continue
            
            seq = resample_sequence(landmarks, SEQUENCE_LENGTH)
            if seq.shape != (SEQUENCE_LENGTH, NUM_FEATURES):
                skipped += 1
                continue
            
            all_X.append(seq)
            all_y.append(class_id)
            
            # Try to extract signer_id from filename
            # KArSL format often: signerX_signY_repZ.mp4
            fname = vpath.stem
            parts = fname.split('_')
            signer_id = parts[0] if len(parts) >= 2 else 'unknown'
            all_signer_ids.append(signer_id)

X = np.array(all_X, dtype=np.float32)
y = np.array(all_y, dtype=np.int32)

print(f'\nDataset built!')
print(f'  X shape: {X.shape}  (sequences, frames, features)')
print(f'  y shape: {y.shape}')
print(f'  Classes: {len(np.unique(y))}')
print(f'  Unique signers: {len(set(all_signer_ids))}')
print(f'  Skipped: {skipped} videos')

# Save NPZ (include signer_ids for merger's signer-aware splitting)
signer_arr = np.array(all_signer_ids)
np.savez_compressed(OUTPUT_NPZ, X=X, y=y, signer_ids=signer_arr)
print(f'\nSaved: {OUTPUT_NPZ}')
print(f'  File size: {OUTPUT_NPZ.stat().st_size / 1024 / 1024:.1f} MB')


## 6. Verify Dataset

In [ ]:
# ============================================================
# CELL 6: QUALITY CHECK
# ============================================================
data = np.load(OUTPUT_NPZ, allow_pickle=True)
X_check, y_check = data['X'], data['y']

print('DATASET VERIFICATION')
print(f'  Shape: X={X_check.shape}, y={y_check.shape}')
print(f'  NaN count: {np.isnan(X_check).sum()}')
print(f'  Feature range: [{X_check.min():.4f}, {X_check.max():.4f}]')
print(f'  Classes: {len(np.unique(y_check))}')

counts = np.bincount(y_check)
counts = counts[counts > 0]  # remove zeros
print(f'  Samples per class (min/max/mean): {counts.min()}/{counts.max()}/{counts.mean():.1f}')

if 'signer_ids' in data:
    print(f'  Signer IDs: {len(np.unique(data["signer_ids"]))} unique')
print('Verification complete')


## 7. BiLSTM Model Training

Same architecture as the ASL word model (~320K params), independent weights.

In [ ]:
# ============================================================
# CELL 7: TRAIN BiLSTM MODEL
# ============================================================
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Bidirectional, LSTM, Dense, Dropout, BatchNormalization, Input
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)
from tensorflow.keras.utils import to_categorical

# Load data
data = np.load(OUTPUT_NPZ, allow_pickle=True)
X_all, y_all = data['X'], data['y']
num_classes = len(np.unique(y_all))

# Train/val/test split (64/16/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y_all
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_SEED, stratify=y_train
)

y_train_oh = to_categorical(y_train, num_classes)
y_val_oh = to_categorical(y_val, num_classes)
y_test_oh = to_categorical(y_test, num_classes)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Classes: {num_classes}')

# Build BiLSTM
model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, NUM_FEATURES)),
    Bidirectional(LSTM(128, return_sequences=True)),
    BatchNormalization(),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax', dtype='float32'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

callbacks = [
    ModelCheckpoint(str(MODEL_OUTPUT), monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=7,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-7, verbose=1),
]

history = model.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

print('Training complete')


## 8. Evaluation

In [ ]:
# ============================================================
# CELL 8: EVALUATE MODEL
# ============================================================
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

best_model = tf.keras.models.load_model(str(MODEL_OUTPUT))

test_loss, test_acc = best_model.evaluate(X_test, y_test_oh, verbose=0)
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test Loss: {test_loss:.4f}')

y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)
print('\nClassification Report (top classes):')
report = classification_report(y_test, y_pred, output_dict=True)
class_metrics = [(k, v['f1-score'], v['support'])
                 for k, v in report.items() if k.isdigit()]
class_metrics.sort(key=lambda x: x[1], reverse=True)
for cls_id, f1, sup in class_metrics[:20]:
    word = id_to_class.get(int(cls_id), cls_id)
    print(f'  {word:20s} F1={f1:.3f} (n={int(sup)})')

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss'); ax2.legend()
plt.tight_layout(); plt.show()

print('Evaluation complete')


## 9. Quick Inference Test

In [ ]:
# ============================================================
# CELL 9: QUICK INFERENCE TEST
# ============================================================
import json as json_lib

idx = np.random.randint(len(X_test))
sample = X_test[idx:idx+1]
true_label = y_test[idx]

pred = best_model.predict(sample, verbose=0)
pred_label = np.argmax(pred)
pred_conf = pred[0][pred_label]

print(f'True class:  {id_to_class.get(true_label, true_label)}')
print(f'Predicted:   {id_to_class.get(pred_label, pred_label)}')
print(f'Confidence:  {pred_conf:.4f}')
print(f'{"CORRECT" if true_label == pred_label else "INCORRECT"}')

# Save class mapping
mapping_path = PROJECT_DIR / 'arsl_word_labels.json'
with open(mapping_path, 'w', encoding='utf-8') as f:
    json_lib.dump({str(k): v for k, v in id_to_class.items()}, f,
                  indent=2, ensure_ascii=False)
print(f'\nWord labels saved: {mapping_path}')
print('\nPipeline complete! Files produced:')
print(f'  {OUTPUT_NPZ}')
print(f'  {MODEL_OUTPUT}')
print(f'  {mapping_path}')
